# Analisis Batu Empedu dengan Regresi Logistik

Tujuan notebook ini adalah menganalisis faktor yang berhubungan dengan `Gallstone Status` dan membangun model klasifikasi yang dapat dievaluasi secara lebih stabil.

Perbaikan metodologi utama:
- scaling dilakukan setelah train-test split untuk mencegah data leakage;
- uji Chi-Square hanya digunakan untuk variabel kategorikal;
- variabel numerik diuji dengan Mann-Whitney U karena banyak marker klinis memiliki outlier;
- multikolinearitas dicek dengan korelasi dan VIF;
- model inferensial dipisahkan dari model prediksi.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm

from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="Set2")


## 1. Load Data dan Pemeriksaan Awal


In [ ]:
DATA_PATH = "batu empedu.csv"
TARGET = "Gallstone Status"

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
data_quality = pd.DataFrame({
    "rows": [df.shape[0]],
    "columns": [df.shape[1]],
    "duplicates": [df.duplicated().sum()],
    "missing_values": [df.isna().sum().sum()],
})

data_quality


In [ ]:
df.info()


In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(df) * 100).round(2),
})

target_distribution


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x=TARGET, ax=ax)
ax.set_title("Distribusi Gallstone Status")
ax.set_xlabel("Gallstone Status (0 = No, 1 = Yes)")
ax.set_ylabel("Jumlah")
plt.show()


## 2. Statistik Deskriptif


In [ ]:
descriptive_stats = df.describe().T
descriptive_stats.to_csv("Statistika deskripsi.csv")
descriptive_stats


In [ ]:
categorical_cols = [
    "Gender",
    "Comorbidity",
    "Diabetes Mellitus (DM)",
    "Coronary Artery Disease (CAD)",
    "Hyperlipidemia",
]

numeric_cols = [col for col in df.columns if col not in categorical_cols + [TARGET]]

print(f"Jumlah variabel kategorikal: {len(categorical_cols)}")
print(f"Jumlah variabel numerik: {len(numeric_cols)}")


In [ ]:
df[numeric_cols].hist(figsize=(16, 14), bins=20)
plt.suptitle("Distribusi Variabel Numerik", y=1.02)
plt.tight_layout()
plt.show()


## 3. Uji Bivariat


Chi-Square dipakai hanya untuk variabel kategorikal. Untuk variabel numerik, digunakan Mann-Whitney U karena beberapa fitur klinis memiliki outlier besar dan distribusinya tidak selalu normal.


In [ ]:
chi_square_results = []

for col in categorical_cols:
    contingency_table = pd.crosstab(df[col], df[TARGET])
    chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)
    chi_square_results.append({
        "Variable": col,
        "Chi2": chi2_stat,
        "p-value": p_value,
        "Degrees of Freedom": dof,
    })

chi_square_results = pd.DataFrame(chi_square_results).sort_values("p-value")
chi_square_results.to_csv("Uji Chi-Square Kategorikal.csv", index=False)
chi_square_results


In [ ]:
mann_whitney_results = []

for col in numeric_cols:
    group_0 = df.loc[df[TARGET] == 0, col]
    group_1 = df.loc[df[TARGET] == 1, col]
    statistic, p_value = stats.mannwhitneyu(group_0, group_1, alternative="two-sided")

    mann_whitney_results.append({
        "Variable": col,
        "Median Target 0": group_0.median(),
        "Median Target 1": group_1.median(),
        "Mann-Whitney U": statistic,
        "p-value": p_value,
    })

mann_whitney_results = pd.DataFrame(mann_whitney_results).sort_values("p-value")
mann_whitney_results.to_csv("Uji Mann-Whitney Numerik.csv", index=False)
mann_whitney_results


## 4. Korelasi dan Multikolinearitas


In [ ]:
corr_matrix = df.drop(columns=[TARGET]).corr(numeric_only=True)

plt.figure(figsize=(14, 11))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Matriks Korelasi Fitur")
plt.show()


In [ ]:
high_corr_pairs = []
abs_corr = corr_matrix.abs()

for i, col_1 in enumerate(abs_corr.columns):
    for col_2 in abs_corr.columns[i + 1:]:
        corr_value = abs_corr.loc[col_1, col_2]
        if corr_value >= 0.80:
            high_corr_pairs.append({
                "Feature 1": col_1,
                "Feature 2": col_2,
                "Absolute Correlation": corr_value,
            })

high_corr_pairs = pd.DataFrame(high_corr_pairs).sort_values(
    "Absolute Correlation",
    ascending=False,
)
high_corr_pairs.to_csv("Korelasi Tinggi.csv", index=False)
high_corr_pairs


In [ ]:
scaled_for_vif = df.drop(columns=[TARGET]).copy()
scaled_for_vif[numeric_cols] = StandardScaler().fit_transform(scaled_for_vif[numeric_cols])

x_vif = sm.add_constant(scaled_for_vif)
vif_results = pd.DataFrame({
    "Feature": scaled_for_vif.columns,
    "VIF": [
        variance_inflation_factor(x_vif.values, i + 1)
        for i in range(len(scaled_for_vif.columns))
    ],
}).sort_values("VIF", ascending=False)

vif_results.to_csv("VIF Semua Fitur.csv", index=False)
vif_results


## 5. Model Inferensial Regresi Logistik

Model penuh tidak dijadikan model utama karena multikolinearitas tinggi dapat membuat estimasi koefisien tidak stabil dan model gagal konvergen. Model inferensial berikut memakai fitur yang lebih ringkas:
- `Age`
- `Comorbidity`
- `Diabetes Mellitus (DM)`
- `Visceral Fat Rating (VFR)`
- `Vitamin D`

Fitur numerik kontinu distandardisasi, sedangkan indikator biner dan ordinal dibiarkan pada skala aslinya agar odds ratio lebih mudah dibaca.


In [ ]:
inference_features = [
    "Age",
    "Comorbidity",
    "Diabetes Mellitus (DM)",
    "Visceral Fat Rating (VFR)",
    "Vitamin D",
]
scaled_inference_features = [
    "Age",
    "Visceral Fat Rating (VFR)",
    "Vitamin D",
]

X_inference = df[inference_features].copy()
X_inference[scaled_inference_features] = StandardScaler().fit_transform(
    X_inference[scaled_inference_features]
)
X_inference = sm.add_constant(X_inference)
y = df[TARGET]

inference_model = sm.Logit(y, X_inference).fit(disp=0)
print(f"Model converged: {inference_model.mle_retvals['converged']}")
inference_model.summary()


In [ ]:
inference_result = pd.DataFrame({
    "Variable": inference_model.params.index,
    "Coefficient": inference_model.params.values,
    "Standard Error": inference_model.bse.values,
    "Wald Statistic": (inference_model.params / inference_model.bse) ** 2,
    "p-value": inference_model.pvalues.values,
    "Odds Ratio": np.exp(inference_model.params.values),
    "OR CI 2.5%": np.exp(inference_model.conf_int()[0].values),
    "OR CI 97.5%": np.exp(inference_model.conf_int()[1].values),
}).sort_values("p-value")

inference_result.to_csv("Hasil Regresi Logistik Inferensial.csv", index=False)
inference_result


In [ ]:
lr_stat = 2 * (inference_model.llf - inference_model.llnull)
df_lr = int(inference_model.df_model)
lr_p_value = chi2.sf(lr_stat, df_lr)

lr_test_result = pd.DataFrame([{
    "Likelihood Ratio Statistic": lr_stat,
    "Degrees of Freedom": df_lr,
    "p-value": lr_p_value,
    "Pseudo R2": inference_model.prsquared,
}])

lr_test_result.to_csv("Uji Serentak Model Inferensial.csv", index=False)
lr_test_result


In [ ]:
def hosmer_lemeshow_test(y_true, y_prob, groups=10):
    data = pd.DataFrame({"y": y_true, "probability": y_prob})
    data["group"] = pd.qcut(data["probability"], q=groups, duplicates="drop")

    observed = data.groupby("group", observed=False)["y"].agg(["sum", "count"])
    observed.columns = ["observed_1", "total"]
    observed["observed_0"] = observed["total"] - observed["observed_1"]
    observed["expected_1"] = data.groupby("group", observed=False)["probability"].sum()
    observed["expected_0"] = observed["total"] - observed["expected_1"]

    hl_stat = (
        ((observed["observed_0"] - observed["expected_0"]) ** 2) / observed["expected_0"]
        + ((observed["observed_1"] - observed["expected_1"]) ** 2) / observed["expected_1"]
    ).sum()

    df_hl = max(len(observed) - 2, 1)
    p_value = 1 - chi2.cdf(hl_stat, df_hl)
    return hl_stat, df_hl, p_value, observed

fitted_probability = inference_model.predict(X_inference)
hl_stat, hl_df, hl_p, hl_table = hosmer_lemeshow_test(y, fitted_probability)

hosmer_lemeshow_result = pd.DataFrame([{
    "Hosmer-Lemeshow Statistic": hl_stat,
    "Degrees of Freedom": hl_df,
    "p-value": hl_p,
}])

hosmer_lemeshow_result.to_csv("Uji Hosmer-Lemeshow.csv", index=False)
hosmer_lemeshow_result


## 6. Model Prediksi dengan Pipeline

Bagian ini fokus pada performa prediksi. Split dibuat stratified agar proporsi target tetap seimbang, dan scaler hanya di-fit pada data train melalui `Pipeline`.


In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("scale_numeric", StandardScaler(), numeric_cols),
        ("keep_categorical", "passthrough", categorical_cols),
    ],
    remainder="drop",
)

logreg_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

logreg_pipeline.fit(X_train, y_train)

y_pred = logreg_pipeline.predict(X_test)
y_pred_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.3f}")


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])

plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

cm_df


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, y_pred_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "f1", "roc_auc"]

cv_scores = cross_validate(
    logreg_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
)

cv_summary = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "ROC-AUC"],
    "Mean": [
        cv_scores["test_accuracy"].mean(),
        cv_scores["test_f1"].mean(),
        cv_scores["test_roc_auc"].mean(),
    ],
    "Std": [
        cv_scores["test_accuracy"].std(),
        cv_scores["test_f1"].std(),
        cv_scores["test_roc_auc"].std(),
    ],
})

cv_summary.to_csv("Cross Validation Logistic Regression.csv", index=False)
cv_summary


## 7. Feature Importance Model Prediksi

Koefisien berikut berasal dari model prediksi berbasis sklearn. Nilai positif meningkatkan peluang prediksi kelas 1, sedangkan nilai negatif menurunkannya. Untuk fitur numerik, interpretasinya berada pada skala standar.


In [ ]:
feature_names = logreg_pipeline.named_steps["preprocess"].get_feature_names_out()
feature_names = [
    name.replace("scale_numeric__", "").replace("keep_categorical__", "")
    for name in feature_names
]

coefficients = logreg_pipeline.named_steps["model"].coef_[0]
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients),
}).sort_values("Absolute Coefficient", ascending=False)

feature_importance.to_csv("Feature Importance Logistic Regression.csv", index=False)
feature_importance


## 8. Ringkasan Interpretasi

Hal yang perlu ditekankan pada laporan:
- data bersih dan target seimbang;
- model penuh memiliki multikolinearitas tinggi, sehingga model inferensial dibuat lebih ringkas;
- hasil inferensial tidak boleh dibaca sebagai hubungan kausal;
- performa prediksi lebih baik dinilai dari cross-validation daripada satu split saja;
- Logistic Regression cukup kompetitif untuk dataset kecil ini dan lebih mudah dijelaskan dibanding model kompleks.
